In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import xarray as xr
import json
#from joblib import Parallel, delayed
#import multiprocessing
import os
import sys
import csv 
import json
from datetime import datetime
import argparse
sys.path.append('/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder')
import utils as ut
import dpa_ensemble as de
import importlib

In [3]:
####################
### Load my data ###
####################
    # load my data
settings_file_path = "../joint_training/v1_dpa_train_settings.json"

with open(settings_file_path, 'r') as file:
        settings = json.load(file)

print("Top-level keys:", list(settings.keys()))

# Load LE temperature data
# Train data
trefht_le = xr.open_dataset(settings['dataset_trefht'])
print(trefht_le)

# Load LE Z500 
z500_le = xr.open_dataset(settings["dataset_z500"])
if True: #args.standardize_predictors:
    predictors_combined_le, _, _ = ut.standardize_numpy(z500_le.pseudo_pcs.values) 
else:
    predictors_combined_le = z500_le.pseudo_pcs.values #xr.concat([gmt_le_expanded, z500_le.pseudo_pcs], dim="mode")


print(z500_le)

# load LE GMT
#gmt_le_pre = xr.open_dataset(settings["GMT_LE"])
#gmt_le = (gmt_le_pre - gmt_le_pre.mean()) / gmt_le_pre.std()
#print(gmt_le)

# concatenate data 
#gmt_le_expanded = gmt_le.TREFHT.expand_dims(mode=[-1]).T  # add 'mode' dimension with length 1
predictors_combined_le
#gmt_le_expanded

# Load ETH ensemble data
# Test data
trefht_eth = xr.open_dataset(settings['dataset_trefht_eth_transient'])
print(trefht_eth)

z500_eth = xr.open_dataset(settings['dataset_z500_eth_test'])
print(z500_eth)

# Load GMT ETH
#gmt_eth_pre = xr.open_dataset(settings["gmt_eth"])
#gmt_eth = (gmt_eth_pre - gmt_eth_pre.mean()) / gmt_eth_pre.std()

# concatenate data 
#gmt_eth_expanded = gmt_eth.TREFHT.expand_dims(mode=[-1]).T  # add 'mode' dimension with length 1
predictors_combined_eth = z500_eth.pseudo_pcs #xr.concat([gmt_eth_expanded, z500_eth.pseudo_pcs], dim="mode")
predictors_combined_eth

# germany domain 
### Germany ###
    
# coordinates 
ger_lat_min = 48
ger_lat_max = 54
ger_lon_min = 6
ger_lon_max = 15

# cut train data
trefht_le_ger = trefht_le.sel(lat=slice(ger_lat_min, ger_lat_max), lon=slice(ger_lon_min, ger_lon_max))
print(trefht_le_ger)

# cut test data
trefht_eth_ger = trefht_eth.sel(lat=slice(ger_lat_min, ger_lat_max), lon=slice(ger_lon_min, ger_lon_max))
print(trefht_eth_ger)

# calculate weighted means
#weights
weights_ger_pre = np.cos(np.deg2rad(trefht_le_ger["lat"]))
weights_ger = weights_ger_pre / weights_ger_pre.sum()

# training data
trefht_le_ger_mean = trefht_le_ger.TREFHT.weighted(weights_ger).mean(dim=("lat", "lon"))
trefht_le_ger_mean

# test_data
trefht_eth_ger_mean = trefht_eth_ger.TREFHT.weighted(weights_ger).mean(dim=("lat", "lon"))
trefht_eth_ger_mean

print("Predictors train shape:", predictors_combined_le.shape)
print("Predictands train shape:", trefht_le_ger_mean.shape)
print("#######")
print("Predictors train shape:", predictors_combined_eth.shape)
print("Predictands test shape:", trefht_eth_ger_mean.shape)


# split 
## train
X_torch = torch.from_numpy(predictors_combined_le)[:90*4769, :]
y_torch = torch.from_numpy(trefht_le_ger_mean.values)[:90*4769]

## validation
X_val_torch = torch.from_numpy(predictors_combined_le)[90*4769:, :]
y_val_torch = torch.from_numpy(trefht_le_ger_mean.values)[90*4769:]

print("X:", X_torch.shape)
print("y:", y_torch.shape)

Top-level keys: ['output_dir', 'dataset_trefht', 'dataset_z500', 'dataset_z500_eth_test', 'dataset_trefht_eth_transient', 'dataset_trefht_eth_nudged', 'dataset_trefht_eth_nudged_shifted']
<xarray.Dataset> Size: 2GB
Dimensions:  (lat: 32, lon: 32, time: 476900)
Coordinates:
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * time     (time) object 4MB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 2GB ...
<xarray.Dataset> Size: 2GB
Dimensions:     (time: 476900, mode: 1001)
Coordinates:
  * time        (time) object 4MB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
  * mode        (mode) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999 1000
Data variables:
    pseudo_pcs  (time, mode) float32 2GB 941.0 -1.472e+03 ... 0.3043 4.744
<xarray.Dataset> Size: 59MB
Dimensions:  (lat: 32, lon: 32, time: 14307)
Coordinates:
  * lon      (lon)

In [5]:
batch_size=128
# create data loader Temperature
train_dataset = TensorDataset(X_torch, y_torch)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of batches: {len(train_loader)}")

# create test loader Temperature
test_dataset = TensorDataset(X_val_torch, y_val_torch)
test_loader_in = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of batches: {len(test_loader_in)}")

Number of batches: 3354
Number of batches: 373


In [3]:
def load_both_dpa_arrays(path, mask, ds_coords, ens_members, save_path=None, no_epochs="not_specified", climate_list=[]):
    '''
    climate_list: ["cf_gen", "gen"]: list of climates (counterfactual and factual) to laod and save
    
    returns:
        lists containing [counterfactual, factual] arrays
    '''
    list_tensor_list = []
    list_tensor_list_raw = []
    list_stacked = []
    list_stacked_reshaped = []
    list_ds = []

    for climate in climate_list:
        print("Now loading DPA ensemble restored including nans")
        print("Path:", f"{path}{climate}1_te.pt")
        tensor_list_raw = [torch.load(f"{path}{climate}{i}_te.pt", map_location=torch.device('cpu'), weights_only=False) for i in range(1,ens_members+1)] #98 #this is loading raw arrays without nans
        list_tensor_list_raw.append(tensor_list_raw)
        print("Tensor list raw length:", len(tensor_list_raw))
        print("Tensor list raw elements shape:", tensor_list_raw[0].shape)
        stacked_raw = torch.stack(tensor_list_raw)  # shape: (N, T, H, W)
        stacked_reshaped_raw = stacked_raw.reshape(stacked_raw.shape[0], stacked_raw.shape[1], stacked_raw.shape[2])
        print((stacked_raw.shape[0], stacked_raw.shape[1], stacked_raw.shape[2]))
    
    
        
        # create dataset
        ds_raw = xr.Dataset({
                        "TREFHT": xr.DataArray(stacked_reshaped_raw.detach().numpy(), 
                             dims=("ensemble_member", "time", "lat_x_lon"),
                             coords={
                                    "ensemble_member": np.arange(1,stacked_reshaped_raw.shape[0]+1),
                                    "time": ds_coords.time,
                                    "lat_x_lon": np.arange(stacked_raw.shape[2])
                                    },
                                              )
        })
        print(ds_raw)
        if save_path is not None:
            # save to zarr
            #ds_raw.to_zarr(f"{save_path}/raw_dpa_ens_100_dataset_restored.zarr", consolidated=True, encoding={"TREFHT": {"_FillValue": None}})
            ds_raw.to_netcdf(f"{save_path}/raw_ETH_{climate}_dpa_ens_{no_epochs}_dataset.nc", format="NETCDF4")
            
        ### Restore NaNs ###
        
        # Load and stack into one tensor
        print("Now loading DPA ensemble restored including nans")
        tensor_list = [restore_nan_columns(torch.load(f"{path}{climate}{i}_te.pt", map_location=torch.device('cpu'), weights_only=False), mask) for i in range(1,ens_members+1)] #98 #this is loading raw arrays without nans
        list_tensor_list.append(tensor_list)
        print("Tensor list length:", len(tensor_list))
        print("Tensor list elements shape:", tensor_list[0].shape)
        stacked = torch.stack(tensor_list)  # shape: (N, T, H, W)
        list_stacked.append(stacked)
        stacked_reshaped = stacked.reshape(stacked.shape[0], stacked.shape[1], 32, 32)
        list_stacked_reshaped.append(stacked_reshaped)
    
    
        
        # create dataset
        ds = xr.Dataset({
                        "TREFHT": xr.DataArray(stacked_reshaped.detach().numpy(), 
                             dims=("ensemble_member", "time", "lat", "lon"),
                             coords={
                                    "ensemble_member": np.arange(1,stacked_reshaped.shape[0]+1),
                                    "time": ds_coords.time,
                                    "lat": ds_coords.lat,
                                    "lon": ds_coords.lon
                                    },
                                              )
        })
        list_ds.append(ds)
        print(ds)
        if save_path is not None:
            # save to zarr
            #ds.to_zarr(f"{save_path}/dpa_ens_100_dataset_restored.zarr", consolidated=True, encoding={"TREFHT": {"_FillValue": None}})
            ds.to_netcdf(f"{save_path}/ETH_{climate}_dpa_ens_{no_epochs}_dataset_restored.nc", format="NETCDF4")
    

    return list_tensor_list, list_tensor_list_raw, list_stacked, list_stacked_reshaped, list_ds

def restore_nan_columns(reduced_tensor: torch.Tensor, column_mask: torch.Tensor):
    """
    Restores removed NaN-only columns to a tensor using the original column mask.
    
    Returns:
        - restored_tensor: Tensor with original number of columns (NaNs in removed positions)
    """
    if reduced_tensor.ndim == 1:
        n_rows = 1
    elif reduced_tensor.ndim > 1:
        n_rows = reduced_tensor.shape[0]
    #n_rows = reduced_tensor.shape[0]
    n_cols = column_mask.numel()
    
    restored_tensor = torch.full((n_rows, n_cols), float('nan'), dtype=reduced_tensor.dtype, device=reduced_tensor.device)
    restored_tensor[:, column_mask] = reduced_tensor
    return restored_tensor

In [9]:
importlib.reload(de)

ens_members=100
save_path_ensemble_single = "/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/ger_1d_dpa/1d_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue"
encoder="learnable"
in_dim=648
latent_dim=50
num_layers=6
hidden_dim=50
bn=True
out_act=None
resblock=True
noise_dim_dec=5
in_dim_lm=1001
num_layers_lm=2
hidden_dim_lm=50
noise_dim_lm=20
model_path="/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/ger_1d_dpa/1d_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue"
epoch_model=4
settings_file_path="v1_dpa_train_settings.json"


os.makedirs(save_path_ensemble_single, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

mask, ds_train, ds_test, x_te_reduced = de.create_ensemble_1d(ensemble_type="ETH",
                                    ensemble_size=ens_members,
                                    save_path=save_path_ensemble_single,
                                    device=device,
                                    encoder=encoder,
                                    in_dim=in_dim,
                                    latent_dim=latent_dim,
                                    num_layers=num_layers,
                                    hidden_dim=hidden_dim,
                                    bn=bn,
                                    out_act=out_act,
                                    resblock=resblock, 
                                    noise_dim_dec=noise_dim_dec,
                                    in_dim_lm=in_dim_lm,
                                    num_layers_lm=num_layers_lm,
                                    hidden_dim_lm=hidden_dim,
                                    noise_dim_lm=noise_dim_lm,
                                    encoder_path=f"{model_path}/model_enc_{epoch_model}.pt",
                                    decoder_path=f"{model_path}/model_dec_{epoch_model}.pt",
                                    lm_path=f"{model_path}/model_pred_{epoch_model}.pt",
                                    settings_file_path=settings_file_path,
                                    create_factual_ensemble=True,
                                    create_counterfactual_ensemble=True,
                                    autoencode=False,
                                    standardize_predictors=True
                                    )

device: cuda
Autoencode: False
torch.Size([14307, 1024])
torch.Size([14307, 1024])
ATTENTION: Z500 PC time-series is standardized manually here
z500 dataset shape (14307, 1001)
Data loaded
device: cuda
DPA model created
Total encoder parameters: 78,150
Total decoder parameters: 14,607
Total latent map parameters: 104,850
Normal predictions ...
Normal cf predictions ...


In [10]:
tensor_list, tensor_list_raw, stacked, stacked_reshaped, ds = load_both_dpa_arrays(path=f"{save_path_ensemble_single}/",
                                                                    mask=mask,
                                                                    ds_coords=ds_test,
                                                                    ens_members=ens_members,
                                                                    save_path=f"{save_path_ensemble_single}",
                                                                    no_epochs=epoch_model,
                                                                    climate_list=["gen", "cf_gen"]
                                                                    )

Now loading DPA ensemble restored including nans
Path: /work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/ger_1d_dpa/1d_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue/gen1_te.pt
Tensor list raw length: 100
Tensor list raw elements shape: torch.Size([14307, 1])
(100, 14307, 1)
<xarray.Dataset> Size: 6MB
Dimensions:          (ensemble_member: 100, time: 14307, lat_x_lon: 1)
Coordinates:
  * ensemble_member  (ensemble_member) int64 800B 1 2 3 4 5 ... 96 97 98 99 100
  * time             (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 0...
  * lat_x_lon        (lat_x_lon) int64 8B 0
Data variables:
    TREFHT           (ensemble_member, time, lat_x_lon) float32 6MB -5.45 ......
Now loading DPA ensemble restored including nans


NameError: name 'restore_nan_columns' is not defined

In [11]:
oned_dpa_ens = xr.open_dataset("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/ger_1d_dpa/1d_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue/raw_ETH_gen_dpa_ens_4_dataset.nc")
oned_dpa_ens

<xarray.Dataset> Size: 6MB
Dimensions:          (ensemble_member: 100, time: 14307, lat_x_lon: 1)
Coordinates:
  * ensemble_member  (ensemble_member) int64 800B 1 2 3 4 5 ... 96 97 98 99 100
  * time             (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 0...
  * lat_x_lon        (lat_x_lon) int64 8B 0
Data variables:
    TREFHT           (ensemble_member, time, lat_x_lon) float32 6MB ...